# ===========================================
### repex_topology_parser
#### Currently only supports Amber Potential 
#### without CMAP corrections
$$ E_{\rm total} = \sum_{\rm bonds} K_r (r - r_{eq})^2 
                     + \sum_{\rm angles} K_\theta (\theta - \theta_{eq})^2
                     + \sum_{\rm dihedrals} {V_n \over 2} 
                                       [1 + {\rm cos}(n\phi - \gamma)]\\ 
                     + \sum_{i<j} \epsilon_{ij} \left [ {\left(\frac{\sigma_{ij}}{R_{ij}}\right)}^{12} - 
                                          {\left(\frac{\sigma_{ij}}{R_{ij}}\right)}^6 \right]
                     + \sum_{i<j} {q_iq_j \over \epsilon R_{ij}} 
                                 
                                 $$
### TODO
#### - Add CMAP lambda scaling
#### - Extend to CHARMM, OPLS-AA
### Examples for REST2, and ssREST2 (solvent-scaled REST2)
# ===========================================

# ===========================================
### Energy Components
$$ E_{\rm total} = E_{\rm protein-protein} + E_{\rm protein-water} + E_{\rm water-water} $$
#### REST2 scaling with $\lambda$
$$ \lambda = \frac{\beta_n}{\beta_0} = \frac{T_0}{T_n} : \beta_n = \frac{1}{k_B T_n} $$
$$ E_{\rm total}^{\lambda_n} = \lambda_n E_{\rm protein-protein} + \sqrt{\lambda_n} E_{\rm protein-water} + E_{\rm water-water} $$
### solvent-scaling 
$$ \kappa_i =  e^{\frac{i}{N - 1} log(\kappa_{max})} ; i \in [0,N-1]$$
$$ E_{\rm total}^{\kappa_n} = E_{\rm protein-protein} + \kappa_n E_{\rm protein-water} + E_{\rm water-water} $$
### ssREST2 (solvent-scaled REST2)
$$ E_{\rm total}^{\lambda_n,\kappa_n} = \lambda_nE_{\rm protein-protein} + \boldsymbol{\kappa_n} \sqrt{\lambda_n} E_{\rm protein-water} + E_{\rm water-water} $$

# ===========================================

In [1]:
# Load our library
import src.repex_topology_parser as rtp

In [252]:
# initialize our class providing an input processed.top file
test_module = rtp.topo2rest('/home/koreyr/github/repex_topology_parser/tests/topology_files/test_topo/processed.top')

In [254]:
### Let us display our molecules contained in our topology
test_module.molecules

{0: 'Protein_chain_A', 1: 'HxD', 2: 'SOL', 3: 'NA', 4: 'CL'}

### Here we perform with one command rest2 scaling. The hot molecule will have dihedrals/charges/LJ 
### parameters scaled by $\sqrt\lambda$. Thus hot molecule-hot molecule interactions will be scaled
### by $\lambda$, while hot molecule - other molecules will be scaled by $\sqrt\lambda$

In [255]:
# Our first example is performing solute scaling (REST2) on just the protein
REST2 = { 'hot_molecules':[0], # Select the molecule(s) you desire to scale, as a list
            'nreps':20, # define the number of replicas you desire
            'outfile':'topol_rest2', # provide a prefix name for your topology
                                     # default = 'topol'
            'filepath':'./test_run/', # define the directory you wish to write your scaled topologies
                                      # default='./'
            'method':'rest2', # method is rest2
            'temps':[300,500], # temperature range 300 to 500, utilized to compute the geometric
                               # temperature ladder
                               # default = [300,500]
            'verbose':True     # be verbose, important if you are performing scaling interactively
                               # e.g. not providing one or more of these options
            }
test_module.run(**REST2)

Running rest2 scaling method


### Here we perform with one command rest2 scaling with an additional scaling on the OW atom of our water model 
### In this case \'OW_tip4pd\' atomtype. First, the input of a hot molecule has the same effect as rest2 where the 
### protein dihedrals/charges/LJ parameters are scaled by $\lambda$, second the LJ $\epsilon$ of water is scaled 
### by $\kappa^2$ tuning the solvation of the hot molecule(s), and lastly, all non-hot molecule-water LJ 
### parameters are reset thus avoiding the effects of solvent scaling. 

In [15]:
import numpy as np

In [256]:
test_string = test_module._scaled_dihedral_types[1.0][:2]

In [257]:
def dihedraltype_array(dihedraltypes:list,prepend:str='',avoid:list=[]):
   data = []
   dtype = np.dtype([('i','U10'),
                  ('j', 'U10'),  # 2nd string
                  ('k', 'U10'),  # 3rd string
                  ('l', 'U10'),  # 4th string
                  ('func', 'i4'),    # 5th integer
                  ('angle', 'f4'),    # 6th float
                  ('K', 'f4'),    # 7th float
                  ('mult', 'i4')])   # 8th integer])
   for string in dihedraltypes:
      line = string[:string.find(';')].split()
      line[0] = prepend + line[0] if line[0] not in avoid else line[0]
      line[3] = prepend + line[3] if line[3] not in avoid else line[3]
      data.append((line[0], line[1], line[2], line[3], int(line[4]), float(line[5]), float(line[6]), int(line[7])))
   return np.array(data, dtype=dtype)

In [258]:
dht = dihedraltype_array([i for i in test_module._sections['dihedraltypes'] if len(i[:i.find(';')].split())>4])

In [259]:
dihedrals = test_module._sections['moleculetype'][0]['dihedrals']
atom_map = test_module.molecule_atoms[0]

In [241]:
np.isin([1,2,3,4],4)

array([False, False, False,  True])

In [247]:
print(~np.array([True,False,False,False]) * np.array([False,True,True,False]) )
print(~np.array([True,False,False,False]) * ~np.array([False,True,True,False]) * np.array([False,False,True,True]))

[False  True  True False]
[False False False  True]


In [282]:
def dihedral_array(dihedraltypes:np.ndarray, dihedrals:list, atom_mapping:dict):
   data = []
   search_dihedrals = []
   dtype_dihedrals = np.dtype([('i','i4'),
                  ('j', 'i4'),  # 2nd string
                  ('k', 'i4'),  # 3rd string
                  ('l', 'i4'),  # 4th string
                  ('func', 'i4'),    # 5th integer
                  ('angle', 'f4'),    # 6th float
                  ('K', 'f4'),    # 7th float
                  ('mult', 'i4')])   # 8th integer])
   dtype = np.dtype([('i','U10'),
                  ('j', 'U10'),  # 2nd string
                  ('k', 'U10'),  # 3rd string
                  ('l', 'U10'),  # 4th string
                  ('func', 'U10'),    # 5th integer
                  ('angle', 'U10'),    # 6th float
                  ('K', 'U10'),    # 7th float
                  ('mult', 'U10')])   # 8th integer])
   for string in dihedrals:
      line = string[:string.find(';')].split()
      if len(line) == 5:
         search_dihedrals.append(tuple(map(int,line)))
      elif len(line) > 5:
         data.append((int(line[0]), int(line[1]), int(line[2]), int(line[3]), int(line[4]), float(line[5]), float(line[6]), int(line[7])))
      elif not line:
         continue
      else:
         print(f'line not processed: {line}')
   for d_quadruple in search_dihedrals:
      if d_quadruple[4] == 4:
         mask = (np.isin(dihedraltypes['i'], atom_mapping[d_quadruple[0]]) & \
               np.isin(dihedraltypes['j'], atom_mapping[d_quadruple[1]]) & \
               np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[2]]) & \
               np.isin(dihedraltypes['l'], atom_mapping[d_quadruple[3]]) & \
               np.isin(dihedraltypes['func'], d_quadruple[4])) | \
               (np.isin(dihedraltypes['i'], atom_mapping[d_quadruple[3]]) & \
               np.isin(dihedraltypes['j'], atom_mapping[d_quadruple[2]]) & \
               np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[1]]) & \
               np.isin(dihedraltypes['l'], atom_mapping[d_quadruple[0]]) & \
               np.isin(dihedraltypes['func'], d_quadruple[4]))
         # mask_Xl = (np.isin(dihedraltypes['i'], 'X') & \
         #       np.isin(dihedraltypes['j'], atom_mapping[d_quadruple[1]]) & \
         #       np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[2]]) & \
         #       np.isin(dihedraltypes['l'], atom_mapping[d_quadruple[3]]) & \
         #       np.isin(dihedraltypes['func'], d_quadruple[4])) 
         # mask_Xll = (np.isin(dihedraltypes['i'], 'X') & \
         #       np.isin(dihedraltypes['j'], 'X') & \
         #       np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[2]]) & \
         #       np.isin(dihedraltypes['l'], atom_mapping[d_quadruple[3]]) & \
         #       np.isin(dihedraltypes['func'], d_quadruple[4]))
         
         for entries in dihedraltypes[mask]:
            data.append((d_quadruple[0], d_quadruple[1], d_quadruple[2], d_quadruple[3], d_quadruple[4], entries[5], entries[6], entries[7])) 
         # for entries in dihedraltypes[~mask*mask_Xl]:
         #    data.append((d_quadruple[0], d_quadruple[1], d_quadruple[2], d_quadruple[3], d_quadruple[4], entries[5], entries[6], entries[7])) 
         # for entries in dihedraltypes[~mask*~mask_Xl*mask_Xll]:
         #    data.append((d_quadruple[0], d_quadruple[1], d_quadruple[2], d_quadruple[3], d_quadruple[4], entries[5], entries[6], entries[7]))
            
      if d_quadruple[4] == 9:
         mask = (np.isin(dihedraltypes['i'], atom_mapping[d_quadruple[0]]) & \
               np.isin(dihedraltypes['j'], atom_mapping[d_quadruple[1]]) & \
               np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[2]]) & \
               np.isin(dihedraltypes['l'], atom_mapping[d_quadruple[3]]) & \
               np.isin(dihedraltypes['func'], d_quadruple[4])) | \
               (np.isin(dihedraltypes['i'], atom_mapping[d_quadruple[3]]) & \
               np.isin(dihedraltypes['j'], atom_mapping[d_quadruple[2]]) & \
               np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[1]]) & \
               np.isin(dihedraltypes['l'], atom_mapping[d_quadruple[0]]) & \
               np.isin(dihedraltypes['func'], d_quadruple[4]))
         mask_Xlr = (np.isin(dihedraltypes['i'], 'X') & \
               np.isin(dihedraltypes['j'], atom_mapping[d_quadruple[1]]) & \
               np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[2]]) & \
               np.isin(dihedraltypes['l'], 'X') & \
               np.isin(dihedraltypes['func'], d_quadruple[4])) | \
               (np.isin(dihedraltypes['i'], 'X') & \
               np.isin(dihedraltypes['j'], atom_mapping[d_quadruple[2]]) & \
               np.isin(dihedraltypes['k'], atom_mapping[d_quadruple[1]]) & \
               np.isin(dihedraltypes['l'], 'X') & \
               np.isin(dihedraltypes['func'], d_quadruple[4]))
      for entries in dihedraltypes[mask]:
         data.append((d_quadruple[0], d_quadruple[1], d_quadruple[2], d_quadruple[3], d_quadruple[4], entries[5], entries[6], entries[7])) 
      for entries in dihedraltypes[~mask*mask_Xlr]:
         data.append((d_quadruple[0], d_quadruple[1], d_quadruple[2], d_quadruple[3], d_quadruple[4], entries[5], entries[6], entries[7])) 
      
   
   structured_array = np.array(data, dtype=dtype_dihedrals)
   structured_array['func'] *= -1
   structured_array.sort(order=['l','k','j','i'])
   structured_array.sort(order='func')
   structured_array['func'] *= -1
   
   return structured_array.astype(dtype)
   

In [283]:
dh_array = dihedral_array(dht, dihedrals, atom_map)

In [284]:
len(dh_array)

3165

# Should have 2649 dihedrals

In [285]:
i,j,k,l = 2,1,5,6
dh_array[(np.isin(dh_array['i'], i)) & (np.isin(dh_array['j'], j)) & (np.isin(dh_array['k'], k) & (np.isin(dh_array['l'], l)))]

array([],
      dtype=[('i', '<U10'), ('j', '<U10'), ('k', '<U10'), ('l', '<U10'), ('func', '<U10'), ('angle', '<U10'), ('K', '<U10'), ('mult', '<U10')])

In [278]:
np.unique(dh_array,axis=-1).shape

(3147,)

In [264]:
dht[(np.isin(dht['i'], 'N')) & (np.isin(dht['j'], 'CT')) & (np.isin(dht['k'], 'C') & (np.isin(dht['l'], 'N')))]

array([('N', 'CT', 'C', 'N', 9, 360., 0.82425, 1),
       ('N', 'CT', 'C', 'N', 9, 180., 6.04588, 2),
       ('N', 'CT', 'C', 'N', 9, 180., 2.00414, 3),
       ('N', 'CT', 'C', 'N', 9, 360., 0.07991, 4),
       ('N', 'CT', 'C', 'N', 9, 360., 0.01674, 5)],
      dtype=[('i', '<U10'), ('j', '<U10'), ('k', '<U10'), ('l', '<U10'), ('func', '<i4'), ('angle', '<f4'), ('K', '<f4'), ('mult', '<i4')])

In [266]:
dh_array[(np.isin(dh_array['i'], 30)) & (np.isin(dh_array['j'], 29)) & (np.isin(dh_array['k'], 35) & (np.isin(dh_array['l'], 37)))]

array([],
      dtype=[('i', '<U10'), ('j', '<U10'), ('k', '<U10'), ('l', '<U10'), ('func', '<U10'), ('angle', '<U10'), ('K', '<U10'), ('mult', '<U10')])

In [271]:
dht[(np.isin(dht['i'], 'N')) & (np.isin(dht['j'], 'C')) & (np.isin(dht['k'], 'CT') & (np.isin(dht['l'], 'H1')))]

array([],
      dtype=[('i', '<U10'), ('j', '<U10'), ('k', '<U10'), ('l', '<U10'), ('func', '<i4'), ('angle', '<f4'), ('K', '<f4'), ('mult', '<i4')])

In [270]:
dht[(np.isin(dht['i'], 'X')) & (np.isin(dht['j'], 'C')) & (np.isin(dht['k'], 'CT') & (np.isin(dht['l'], 'X')))]

array([('X', 'C', 'CT', 'X', 9, 0., 0., 0)],
      dtype=[('i', '<U10'), ('j', '<U10'), ('k', '<U10'), ('l', '<U10'), ('func', '<i4'), ('angle', '<f4'), ('K', '<f4'), ('mult', '<i4')])

In [237]:
[' '.join(line)+'\n' for line in dh_array]

['5 7 10 13 9 0.0 0.6276 3\n',
 '7 5 23 25 9 0.0 0.8368 1\n',
 '7 5 23 25 9 0.0 0.8368 2\n',
 '7 5 23 25 9 0.0 1.6736 3\n',
 '7 10 13 16 9 0.0 0.6276 3\n',
 '8 7 10 11 9 0.0 0.6276 3\n',
 '8 7 10 12 9 0.0 0.6276 3\n',
 '8 7 10 13 9 0.0 0.66944 3\n',
 '9 7 10 11 9 0.0 0.6276 3\n',
 '9 7 10 12 9 0.0 0.6276 3\n',
 '9 7 10 13 9 0.0 0.66944 3\n',
 '11 10 13 14 9 0.0 0.6276 3\n',
 '11 10 13 15 9 0.0 0.6276 3\n',
 '11 10 13 16 9 0.0 0.66944 3\n',
 '12 10 13 14 9 0.0 0.6276 3\n',
 '12 10 13 15 9 0.0 0.6276 3\n',
 '12 10 13 16 9 0.0 0.66944 3\n',
 '23 25 27 33 9 0.0 0.33472 4\n',
 '23 25 27 33 9 0.0 1.40164 2\n',
 '23 25 27 33 9 0.0 2.2761 3\n',
 '23 25 27 33 9 180.0 0.14226 1\n',
 '25 27 33 35 9 180.0 2.00414 3\n',
 '25 27 33 35 9 180.0 6.04588 2\n',
 '25 27 33 35 9 360.0 0.01674 5\n',
 '25 27 33 35 9 360.0 0.07991 4\n',
 '25 27 33 35 9 360.0 0.82425 1\n',
 '28 27 33 34 9 0.0 3.3472 1\n',
 '28 27 33 34 9 180.0 0.33472 3\n',
 '29 27 33 35 9 0.0 0.8368 1\n',
 '29 27 33 35 9 0.0 0.8368 2\n',
 '29

In [ ]:
dh_array.astype()

In [6]:
test_module._get_molecule_atomtypes(2)

array(['HW', 'MW', 'OW_tip4pd'], dtype=object)

In [ ]:
# Our second example is performing solvent-scaling REST3 (ssREST3) on just the protein
# From the displayed atomtypes contained in our solvent molecule we opt to scale
# 'OW_tip4pd' ('HW' and 'MW' have epsilon = 0.0 so we exclude these from the list)
ssREST3 = { 'hot_molecules':[0], # Select the molecule(s) you desire to scale, as a list
            'nreps':20, # define the number of replicas you desire
            'outfile':'topol_ssrest3', # provide a prefix name for your topology
                                     # default = 'topol'
            'filepath':'./', # define the directory you wish to write your scaled topologies
                                      # default='./'
            'method':'ssrest3', # method is ssrest3
            'temps':[300,500], # temperature range 300 to 500, utilized to compute the geometric
                               # temperature ladder
                               # default = [300,500]
            'kappa_low_temp' : 300, # At which temperature to activate solvent scaling, maybe useful
                                    # to increse to 330 when using a lower number of replicas so the base replica
                                    # experiences solvation more accurately. For ssREST3 simulations with 
                                    # 16 or more replicas, it is unlikedly changing this value to 330 will have 
                                    # any benefit. 
            'kappa_max' : 1.1, # Maximum kappa value
            'kappa_atom_names' : ['OW_tip4pd'], # List of atomtypes to apply kappa scaling
            'verbose':True     # be verbose, important if you are performing scaling interactively
                               # e.g. not providing one or more of these options
            }
test_module.run(**ssREST3)

Running ssrest3 scaling method


In [8]:
import numpy as np
np.arange(1.02,1.1,0.01)

array([1.02, 1.03, 1.04, 1.05, 1.06, 1.07, 1.08, 1.09, 1.1 ])

In [10]:
test_module._get_molecule_atomtypes(1)

array(['HW', 'MW', 'OW_tip4pd'], dtype=object)

In [14]:
for k in np.arange(1.02,1.1,0.01):
    ssREST3 = { 'hot_molecules':[0], 
                'nreps':20, 
                'outfile':'topol_ssrest3',          
                'filepath':'./', 
                'method':'ssrest3',
                'temps':[300,500], 
                'kappa_low_temp' : 300, 
                'kappa_max' : k, 
                'kappa_atom_names' : ['OW_tip4pd'], 
                'verbose':True     
                }
    test_module.run(**ssREST3)

Running ssrest3 scaling method
Running ssrest3 scaling method
Running ssrest3 scaling method
Running ssrest3 scaling method
Running ssrest3 scaling method
Running ssrest3 scaling method
Running ssrest3 scaling method
Running ssrest3 scaling method
Running ssrest3 scaling method
